In [46]:
"""Processes GW2 market and recipe data into a unified analytical DataFrame."""

import json
import os
from typing import Any, Dict, List

import pandas as pd

# Configuration
_MARKET_SUMMARY = "/home/jdforres/gw2_tools/market_analysis/data_harvester/processed_data/market_summary.json"
_RECIPES_PROCESSED = "/home/jdforres/gw2_tools/market_analysis/data_harvester/processed_data/recipes_processed.json"

_MIN_DAILY_SOLD = 10  # Minimum daily velocity to be considered relevant


class MarketAnalyticEngine:
  """Joins and flattens market data layers for analysis."""

  def __init__(self):
    self.market_data = self._load_json(_MARKET_SUMMARY)
    self.recipe_data = self._load_json(_RECIPES_PROCESSED)

  def _load_json(self, path: str) -> Any:
    if not os.path.exists(path):
      print(f"⚠️ Warning: File not found at {path}")
      return {}
    with open(path, 'r', encoding='utf-8') as f:
      return json.load(f)

  def create_dataframe(self) -> pd.DataFrame:
    """Joins metadata, market, and recipes into a single DataFrame."""
    
    # 1. Start with the Market Summary
    # We convert the dict to a list of records for Pandas
    market_records = []
    for item_id, metrics in self.market_data.items():
      record = {"item_id": int(item_id)}
      record.update(metrics)
      # Calculate Flip Profit explicitly
      record['flip_profit'] = int(metrics['sell'] * 0.85) - (metrics['buy'] + 1)
      market_records.append(record)
    
    df_market = pd.DataFrame(market_records)

    # 2. Process Recipes (Finding the best/cheapest recipe per item)
    recipe_records = []
    for r in self.recipe_data:
      recipe_records.append({
          "item_id": r['output_item_id'],
          "craft_profit": r['profit'],
          "craft_roi": r['roi'],
          "craft_cost": r['craft_cost'],
          "recipe_id": r['recipe_id'],
          "recipe_liquidity": r['lowest_ing_l_ratio']
      })
    
    df_recipes = pd.DataFrame(recipe_records)
    # If multiple recipes exist for one item, keep the one with the highest profit
    df_recipes = df_recipes.sort_values('craft_profit', ascending=False).drop_duplicates('item_id')

#     # 3. Process Metadata
#     meta_records = [{"item_id": int(k), "name": v['name'], "type": v['type']} 
#                     for k, v in self.metadata.items()]
#     df_meta = pd.DataFrame(meta_records)

    # 4. Join the layers (Left join on Market to keep all tradable items)
#     final_df = df_market.merge(df_meta, on='item_id', how='left')
    final_df = df_market.merge(df_recipes, on='item_id', how='left')

    # Fill NaNs for items without recipes
    final_df['craft_profit'] = final_df['craft_profit'].fillna(0).astype(int)
    final_df['craft_roi'] = final_df['craft_roi'].fillna(0.0)

    return final_df

In [47]:
import numpy as np

engine = MarketAnalyticEngine()
df = engine.create_dataframe().set_index('item_id')

df['flip_roi'] = round(df['flip_profit'] / df['buy'], 2)
df['daily_market_cap_g'] = round(np.minimum(df['daily_sold_est'], df["daily_bought_est"]) * df['flip_profit'] / 10000)
df['daily_craft_market_cap_g'] = round(df['daily_sold_est'] * df['craft_profit'] * df['recipe_liquidity'] / 10000)
df['daily_craft_market_cap_g_il'] = round(df['daily_sold_est'] * df['craft_profit'] / 10000)

In [48]:
from itables import show

min_daily_sales = 100

condition = (
    (
        True
        & (df['craft_roi'] > 0.25) 
        & (df['craft_profit'] > 100)
    )
    & (df['daily_sold_est'] >= min_daily_sales)
    & (df['recipe_liquidity'] >= 0.3)
)

# condition = (
#     (
#         (df['sell'] > 500)
#     )
#     & (df['daily_sold_est'] >= min_daily_sales)
#     & (df['daily_market_cap_g'] >= 60)
# #     & (df['flip_roi'] > 0.1)
# #     & (df['buy'] > 0)
# )

final_df = df[condition].copy()
final_df = df.copy()

show(final_df, maxBytes=0)

In [49]:

# gray dyes yield black + white pigment
gray_dyes = [
  "Permafrost Dye",
  "Eternal Ice Dye",
  "Celestial Dye",
  "Crushed Bone Dye",
  "White Dye",
  "Spirits Dye",
  "Tarnished Silver Dye",
  "Icing Dye",
  "Fog Dye",
  "Silver Satin Dye",
  "Silver Dye",
  "Frost Dye",
  "Risen Dye",
  "Green Tint Dye",
  "Blue Tint Dye",
  "Fluff Dye",
  "Violet Tint Dye",
  "Gladium Dye",
  "Dust Dye",
  "Stone Dye",
  "Silt Dye",
  "Burnished Steel Dye",
  "Chalk Dye",
  "Dry Silver Dye",
  "Slate Dye",
  "Steel Dye",
  "Ash Dye",
  "Lead Dye",
  "Godless Dye",
  "Gray Dye",
  "Silver Lead Dye",
  "Vintage Silver Dye",
  "Pewter Dye",
  "Graphite Dye",
  "Murky Gray Dye",
  "Darkness Dye",
  "Ebony Dye",
  "Pitch Dye",
  "Night Shade Dye",
  "Iron Dye",
  "Grave Dye",
  "Black Dye",
  "Glossy Black Dye",
  "Abyss Dye",
  "Tar Dye",
  "Shadow Abyss Dye",
  "Midnight Fire Dye"
]


condition = (
    df["name"].isin(gray_dyes)
    & (df['sell'] < 2500)
)

columns_to_remove = [
    'supply',
    'demand',
    'daily_sold_est',
    'daily_bought_est',
    'flip_profit',
    'craft_profit',
    'craft_roi',
    'craft_cost',
    'recipe_id',
    'recipe_liquidity',
    'flip_roi',
    'daily_market_cap_g',
    'daily_craft_market_cap_g',
    'daily_craft_market_cap_g_il',
]

gray_dyes_df = df[condition].copy().drop(columns=columns_to_remove)



# Based on research in https://docs.google.com/spreadsheets/d/1utXSQo4wxznIfLX5sAHVw-OFWRbUJ0I6xzCB-h5HjLg/edit?gid=610475825#gid=610475825
# Fine Gray Dyes -> ~3 pigments (1.5x black, 1.5x white)
# Masterwork Gray Dyes -> ~6.5 pigments (3.25 black, 3.25 white)
# Rare Gray Dyes -> ~10.4 pigments (5.2 black, 5.2 white)

# TODO add expected value/profit to chart.
# Better yet, compute this with the buy/sell orders to know how many of each at what profit.
name_indexed_df = df.set_index("name")
# show(name_indexed_df, maxBytes=0)

black_pigment_price = name_indexed_df.at["Pouch of Black Pigment", "buy"]
white_pigment_price = name_indexed_df.at["Pouch of White Pigment", "buy"]

def expected_profit(row):
    # You can access multiple columns in the same row here
    if row["rarity"] == "Fine":
        return round(0.85 * (1.5 * black_pigment_price + 1.5 * white_pigment_price))
    elif row["rarity"] == "Masterwork":
        return round(0.85 * (3.25 * black_pigment_price + 3.25 * white_pigment_price))
    elif row["rarity"] == "Rare":
        return round(0.85 * (5.2 * black_pigment_price + 5.2 * white_pigment_price))
    else:
        return 0


# You MUST include axis=1 so pandas applies the function row-by-row
gray_dyes_df["expected_value"] = gray_dyes_df.apply(expected_profit, axis=1)
gray_dyes_df["sell_profit"] = gray_dyes_df["expected_value"] - gray_dyes_df["sell"]
gray_dyes_df["sell_profit_roi"] = round(gray_dyes_df["sell_profit"] / gray_dyes_df["sell"], 2)
gray_dyes_df["break_even_50"] = round(gray_dyes_df["expected_value"] * 0.66, 2)
# df['daily_market_cap_g'] = round(df['daily_sold_est'] * df['flip_profit'] / 10000)

show(gray_dyes_df, maxBytes=0)

In [50]:
name_indexed_df = df.set_index("name")

black_pigment_price = name_indexed_df.at["Pouch of Black Pigment", "buy"]
white_pigment_price = name_indexed_df.at["Pouch of White Pigment", "buy"]

